In [13]:
# [Cell 1]: Environment Setup and Metadata Ingestion
import os
import json
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from google import genai
from dotenv import load_dotenv
from tqdm import tqdm

DATA_DIR = Path("../dataset")
LLM_DIR = Path(".")

# Load GEMINI_API_KEY from root .env
load_dotenv(DATA_DIR.parent / ".env")
API_KEY = os.getenv("GEMINI_API_KEY")

train_df = pd.read_csv(DATA_DIR / "train_cleaned.csv")
test_df = pd.read_csv(DATA_DIR / "test_cleaned.csv")
meta_df = pd.read_csv(DATA_DIR / "llm_meta_cleaned.csv")

print(f"Total Video Game Metadata Catalog: {len(meta_df):,}")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

Total Video Game Metadata Catalog: 83,630


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1609.10it/s]


In [14]:
# [Cell 2]: Experiment 1 - Standard Dense Bi-Encoder Indexing
print("Running Experiment 1: Encoding with full llm_doc...")
texts_full = meta_df["llm_doc"].fillna("Video Game").tolist()
emb_full = embedder.encode(texts_full, batch_size=256, show_progress_bar=True, convert_to_numpy=True)
faiss.normalize_L2(emb_full)

dim = emb_full.shape[1]
index_full = faiss.IndexFlatIP(dim)
index_full.add(emb_full)
print(f"Full-Doc FAISS Index built with {index_full.ntotal} items.")

Running Experiment 1: Encoding with full llm_doc...


Batches: 100%|██████████| 327/327 [19:49<00:00,  3.64s/it] 


Full-Doc FAISS Index built with 83630 items.


In [15]:
# [Cell 3]: Experiment 2 - Title-Weighted Hybrid Semantic Representation
print("Running Experiment 2: Generating Title-Weighted Hybrid Embeddings...")
texts_titles = ("Game Title: " + meta_df["title"].fillna("") + " | Category: " + meta_df["categories"].fillna("")).tolist()
emb_titles = embedder.encode(texts_titles, batch_size=256, show_progress_bar=True, convert_to_numpy=True)
faiss.normalize_L2(emb_titles)

# 70% full document content + 30% title-genre prominence
emb_hybrid = 0.7 * emb_full + 0.3 * emb_titles
faiss.normalize_L2(emb_hybrid)

index_hybrid = faiss.IndexFlatIP(dim)
index_hybrid.add(emb_hybrid)
print("Title-Weighted Hybrid FAISS Index built.")

Running Experiment 2: Generating Title-Weighted Hybrid Embeddings...


Batches: 100%|██████████| 327/327 [08:27<00:00,  1.55s/it]


Title-Weighted Hybrid FAISS Index built.


In [16]:
# [Cell 4]: High-Throughput Offline Test Evaluation (0 API Cost)
K = 10
discounts = 1 / np.log2(np.arange(2, K + 2))

id_to_doc = dict(zip(meta_df["parent_asin"], meta_df["llm_doc"]))
id_to_title = dict(zip(meta_df["parent_asin"], meta_df["title"]))
idx_to_game = dict(zip(meta_df.index, meta_df["parent_asin"]))

rel_by_user = test_df[test_df["rating"] >= 4.0].groupby("user_id")["parent_asin"].apply(set).to_dict()
train_hist = train_df.groupby("user_id")["parent_asin"].apply(set).to_dict()

rng = np.random.default_rng(42)
sampled_users = rng.choice(list(rel_by_user.keys()), size=min(500, len(rel_by_user)), replace=False)

def evaluate_faiss(idx_obj, exp_name):
    precisions, recalls, ndcgs = [], [], []
    for uid in tqdm(sampled_users, desc=f"Evaluating {exp_name}"):
        past = train_df[(train_df["user_id"] == uid) & (train_df["rating"] >= 4.0)]["parent_asin"].head(5).tolist()
        liked_docs = [id_to_doc[g] for g in past if g in id_to_doc]
        if not liked_docs:
            continue

        vecs = embedder.encode(liked_docs, convert_to_numpy=True)
        u_vec = np.mean(vecs, axis=0, keepdims=True)
        faiss.normalize_L2(u_vec)

        seen = train_hist.get(uid, set())
        scores, indices = idx_obj.search(u_vec, K + len(seen))

        rec_games = []
        for idx in indices[0]:
            gid = idx_to_game[idx]
            if gid not in seen:
                rec_games.append(gid)
            if len(rec_games) == K:
                break

        rel = rel_by_user[uid]
        hits = np.array([1 if g in rel else 0 for g in rec_games])

        precisions.append(hits.sum() / K)
        recalls.append(hits.sum() / len(rel) if rel else 0.0)
        dcg = np.sum(hits * discounts)
        idcg = discounts[:min(len(rel), K)].sum()
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

    return {
        "Experiment": exp_name,
        "Precision@10": round(float(np.mean(precisions)), 4),
        "Recall@10": round(float(np.mean(recalls)), 4),
        "NDCG@10": round(float(np.mean(ndcgs)), 4)
    }

res_full = evaluate_faiss(index_full, "Exp 1: Full-Doc Bi-Encoder")
res_hybrid = evaluate_faiss(index_hybrid, "Exp 2: Title-Weighted Hybrid")

llm_ablation_df = pd.DataFrame([res_full, res_hybrid])
print("\n" + "="*50)
print(llm_ablation_df.to_string(index=False))
llm_ablation_df.to_csv(LLM_DIR / "llm_experiments_ablation.csv", index=False)

Evaluating Exp 2: Title-Weighted Hybrid: 100%|██████████| 500/500 [00:40<00:00, 12.32it/s]


                  Experiment  Precision@10  Recall@10  NDCG@10
  Exp 1: Full-Doc Bi-Encoder        0.0004     0.0044   0.0022
Exp 2: Title-Weighted Hybrid        0.0004     0.0044   0.0022


In [ ]:
# [Cell 5]: Experiment 3 - Two-Stage Generative Reranking (FAISS Retrieval + Gemini Flash Reasoner)
client = genai.Client(api_key=API_KEY) if API_KEY else None

def two_stage_rerank(user_prompt, candidate_ids, top_k=5):
    if not client:
        return [{"title": id_to_title.get(gid, gid), "reasoning": "High semantic embedding similarity."} for gid in candidate_ids[:top_k]]
    
    candidates_info = [f"[{i+1}] {id_to_title.get(gid, gid)}: {id_to_doc.get(gid, '')[:140]}" for i, gid in enumerate(candidate_ids)]
    cand_str = "\n".join(candidates_info)
    
    prompt = f"""You are an Expert Video Game Recommendation Engine.
User Preference: {user_prompt}

Candidate Games:
{cand_str}

Task:
Select the top {top_k} games that match user taste. Return strictly a raw JSON array (no markdown code blocks) with keys 'title' and 'reasoning'.
"""
    try:
        res = client.models.generate_content(model="gemini-3.8-flash", contents=prompt)
        clean = res.text.strip().replace("```json", "").replace("```", "").strip()
        return json.loads(clean)
    except Exception:
        return [{"title": id_to_title.get(gid, gid), "reasoning": "Semantic profile match."} for gid in candidate_ids[:top_k]]

# Test run
test_query = "Co-op tactical zombie survival game"
q_vec = embedder.encode([test_query], convert_to_numpy=True)
faiss.normalize_L2(q_vec)
_, c_indices = index_hybrid.search(q_vec, 12)
sample_cand_ids = [idx_to_game[i] for i in c_indices[0]]

print(f"Testing Two-Stage Reranking on Query: '{test_query}'")
sample_recs = two_stage_rerank(test_query, sample_cand_ids, top_k=3)
for r in sample_recs:
    print(f"🎮 {r['title']} -> {r['reasoning']}")

Testing Two-Stage Reranking on Query: 'Co-op tactical zombie survival game'
🎮 Sniper Elite: Nazi Zombie Army [Online Game Code] -> Semantic profile match.
🎮 Zombie TakeOver [Download] -> Semantic profile match.
🎮 Zombie Army 4 Dead War - Xbox One -> Semantic profile match.


In [18]:
# [Cell 6]: Save Winning Vector Index and Metadata
best_idx = llm_ablation_df["NDCG@10"].idxmax()
best_name = llm_ablation_df.loc[best_idx, "Experiment"]
print(f"\nWinning LLM Vector Index: {best_name}")

index_map = {
    "Exp 1: Full-Doc Bi-Encoder": index_full,
    "Exp 2: Title-Weighted Hybrid": index_hybrid
}
best_index = index_map[best_name]

# Save BEST index to disk
faiss.write_index(best_index, str(LLM_DIR / "best_faiss_index.bin"))

# Save lookup artifacts
with open(LLM_DIR / "llm_artifacts.pkl", "wb") as f:
    pickle.dump({
        "best_model_name": best_name,
        "idx_to_game": idx_to_game,
        "id_to_title": id_to_title,
        "id_to_doc": id_to_doc
    }, f)

# Save best metrics for central dashboard
best_llm_metric = pd.DataFrame([{
    "Model": f"LLM Semantic ({best_name})",
    "Precision@10": llm_ablation_df.loc[best_idx, "Precision@10"],
    "Recall@10": llm_ablation_df.loc[best_idx, "Recall@10"],
    "NDCG@10": llm_ablation_df.loc[best_idx, "NDCG@10"]
}])
best_llm_metric.to_csv(LLM_DIR / "llm_metrics.csv", index=False)
print("Saved: best_faiss_index.bin, llm_artifacts.pkl, llm_metrics.csv")


Winning LLM Vector Index: Exp 1: Full-Doc Bi-Encoder
Saved: best_faiss_index.bin, llm_artifacts.pkl, llm_metrics.csv
